# FloatDeck V4：独立一键人物覆盖复审
**只点「人物覆盖复审（白纱/裙摆/腿脚）」。** 全新会话或记录丢失时自动准备并重新验证环境，素材丢失时只需上传一次原审核ZIP。
珍珠2固定、人物5保持Reject；不运行Alpha、背景或最终打包。


In [ ]:
#@title ▶ 一键环境检查与诊断（只需点击本单元左侧播放键）
from pathlib import Path
import os,sys,json,subprocess,shutil,re,time,traceback,zipfile
BASE=Path('/content/floatdeck-v4');ROOT=Path('/content/FloatDeck-v4-generator')
ENV_READY=False;ALLOW_ASSET_STAGES=False
bootstrap={'passed':False,'stage':'repository','commands':[],'models_started':False}
def redact(text,env=None):
    text=str(text)
    for k,v in (env or {}).items():
        if re.search(r'token|password|secret|credential|api.?key|authorization',k,re.I) and len(v)>=6:
            text=text.replace(v,'[REDACTED]')
    text=re.sub(r'(https?://)[^\s/@]+:[^\s/@]+@',r'\1[REDACTED]@',text)
    text=re.sub(r'(?i)Bearer\s+[^\s\"\x27]+','Bearer [REDACTED]',text)
    text=re.sub(r'(?i)((?:token|password|secret|api[_-]?key|authorization|signature|credential)[\w-]*[\s\"\x27]*[:=][\s\"\x27]*)([^\s\"\x27,&}]+)',r'\1[REDACTED]',text)
    text=re.sub(r'\b(?:hf_|ghp_|github_pat_)[A-Za-z0-9_]+','[REDACTED]',text)
    return text
def boot(args):
    p=subprocess.run(list(map(str,args)),capture_output=True,text=True,errors='replace')
    bootstrap['commands'].append({'argv':list(map(str,args)),'returncode':p.returncode,'stdout':p.stdout,'stderr':p.stderr})
    if p.returncode:raise subprocess.CalledProcessError(p.returncode,args,p.stdout,p.stderr)
    return p
try:
    BASE.mkdir(parents=True,exist_ok=True)
    print('正在更新生成器并准备独立 Python 环境，请等待；不会启动任何图像模型。')
    if not (ROOT/'.git').exists():
        boot(['git','clone','--branch','spatial-layered-v4-generator','--single-branch','https://github.com/fdsfbgdf-blip/FloatDeck.git',ROOT])
    else:
        branch=boot(['git','-C',ROOT,'branch','--show-current']).stdout.strip()
        if branch!='spatial-layered-v4-generator':raise RuntimeError('已有目录位于其他分支，为保护代码已停止；请发送诊断包。')
        boot(['git','-C',ROOT,'pull','--ff-only','origin','spatial-layered-v4-generator'])
    if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
    # 反复点击时也加载最新辅助代码，不复用旧内核模块。
    import importlib
    import tools.layered_v4.diagnostics as diagnostics
    importlib.reload(diagnostics)
    session=diagnostics.DiagnosticSession(BASE,ROOT)
    session.report['commands'].extend(bootstrap['commands'])
    result=diagnostics.prepare(session)
    ENV_READY=result['passed'];PY=result['python'];WORK=Path(result['work']);MODEL_ENV=result['env']
    DIAGNOSTIC_ZIP=Path(result['zip'])
    from tools.layered_v4.process import run_logged
    def checked(argv):return run_logged(argv,cwd=ROOT,env=MODEL_ENV,log_dir=BASE/'command-logs')
    def run(command,*extra):
        if not ENV_READY:raise RuntimeError('环境尚未通过检查，禁止启动后续任务')
        return checked([PY,'-m','tools.layered_v4.cli',command,'--work',str(WORK),*map(str,extra)])
except Exception as exc:
    bootstrap.update(error=str(exc),traceback=traceback.format_exc(),message='环境准备失败，已停止。请将自动下载的诊断 ZIP 发给我；无需阅读日志或运行命令。')
    DIAGNOSTIC_ZIP=BASE/'FloatDeck-environment-diagnostics.zip'
    with zipfile.ZipFile(DIAGNOSTIC_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('report.json',redact(json.dumps(bootstrap,ensure_ascii=False,indent=2),os.environ))
    print(bootstrap['message'])
finally:
    if 'DIAGNOSTIC_ZIP' in globals() and DIAGNOSTIC_ZIP.exists():
        from IPython.display import display,FileLink
        display(FileLink(str(DIAGNOSTIC_ZIP)))
        try:
            from google.colab import files
            files.download(str(DIAGNOSTIC_ZIP))
        except Exception:print('浏览器未自动下载时，点击上面的 ZIP 链接，或从左侧文件面板下载同名文件。')
print('本次操作结束。诊断 ZIP：FloatDeck-environment-diagnostics.zip；无需执行下面的素材生成单元。')


## 人物覆盖复审（白纱/裙摆/腿脚）
**独立一键入口，无需先运行其他单元。** 自动恢复/重新验证当前Python与GPU环境，复用现有依赖和缓存；素材丢失时只弹一次审核ZIP上传框。珍珠2固定，人物5保持Reject；不运行Alpha/背景/最终打包。


In [ ]:
# 独立环境由共享准备器使用 uv --seed 创建，不依赖系统 ensurepip。
#@title ▶ 人物覆盖复审（白纱/裙摆/腿脚）（独立一键：自动恢复环境和素材）
from pathlib import Path
import os,sys,json,subprocess,shutil,re,time,traceback,zipfile
BASE=Path('/content/floatdeck-v4');ROOT=Path('/content/FloatDeck-v4-generator')
ENV_READY=False;ALLOW_ASSET_STAGES=False
bootstrap={'passed':False,'stage':'repository','commands':[],'models_started':False}
def redact(text,env=None):
    text=str(text)
    for k,v in (env or {}).items():
        if re.search(r'token|password|secret|credential|api.?key|authorization',k,re.I) and len(v)>=6:
            text=text.replace(v,'[REDACTED]')
    text=re.sub(r'(https?://)[^\s/@]+:[^\s/@]+@',r'\1[REDACTED]@',text)
    text=re.sub(r'(?i)Bearer\s+[^\s\"\x27]+','Bearer [REDACTED]',text)
    text=re.sub(r'(?i)((?:token|password|secret|api[_-]?key|authorization|signature|credential)[\w-]*[\s\"\x27]*[:=][\s\"\x27]*)([^\s\"\x27,&}]+)',r'\1[REDACTED]',text)
    text=re.sub(r'\b(?:hf_|ghp_|github_pat_)[A-Za-z0-9_]+','[REDACTED]',text)
    return text
def boot(args):
    p=subprocess.run(list(map(str,args)),capture_output=True,text=True,errors='replace')
    bootstrap['commands'].append({'argv':list(map(str,args)),'returncode':p.returncode,'stdout':p.stdout,'stderr':p.stderr})
    if p.returncode:raise subprocess.CalledProcessError(p.returncode,args,p.stdout,p.stderr)
    return p
CONTINUE_AFTER_SEGMENTATION=False
PERSON_ZIP=None
ENTRY_CALLED=False
try:
    BASE.mkdir(parents=True,exist_ok=True)
    print('正在自动检查/恢复当前环境，之后验证素材；完成前不会启动SAM。')
    if not (ROOT/'.git').exists():
        boot(['git','clone','--branch','spatial-layered-v4-generator','--single-branch','https://github.com/fdsfbgdf-blip/FloatDeck.git',ROOT])
    else:
        if boot(['git','-C',ROOT,'branch','--show-current']).stdout.strip()!='spatial-layered-v4-generator':raise RuntimeError('已有代码位于其他分支，已停止')
        boot(['git','-C',ROOT,'pull','--ff-only','origin','spatial-layered-v4-generator'])
    if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
    import importlib
    importlib.invalidate_caches()
    # 旧会话也必须加载新编排代码；所有这些模块在导入时都只依赖标准库。
    import tools.layered_v4.diagnostics as diagnostics
    import tools.layered_v4.sam_export as sam_export
    import tools.layered_v4.person_review as person_review
    importlib.reload(diagnostics);importlib.reload(sam_export);importlib.reload(person_review)
    import tools.layered_v4.coverage_entry as coverage_entry
    importlib.reload(coverage_entry)
    def upload_review_once():
        print('原审核素材已丢失。请在此唯一上传框选择 FloatDeck-SAM-review.zip 或完整人物/Alpha复审包。')
        from google.colab import files
        return files.upload()
    ENTRY_CALLED=True
    result=coverage_entry.run_coverage(BASE,ROOT,upload_once=upload_review_once,bootstrap_commands=bootstrap['commands'])
    PERSON_ZIP=Path(result['zip'])
    if result['candidate_generation_complete']:
        try:
            from IPython.display import display,Image as DisplayImage
            for path in sorted((Path(result['work'])/'segmentation-previews').glob('person_review_*.jpg')):display(DisplayImage(filename=str(path)))
        except Exception:print('内嵌预览显示失败，但结果ZIP已生成，请直接下载查看。')
    else:print('【未完成分割】本次下载的是失败诊断包，不是六候选结果；请直接发送ZIP。')
except (Exception,KeyboardInterrupt) as exc:
    bootstrap.update(passed=False,candidate_generation_complete=False,models_started=None if ENTRY_CALLED else False,error=str(exc),traceback=traceback.format_exc(),message='【未完成分割：失败诊断包】未取得完整可审核结果。请发送下载ZIP，无需处理路径或Python。')
    PERSON_ZIP=BASE/'FloatDeck-person-coverage-rereview.zip'
    with zipfile.ZipFile(PERSON_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('00_请先阅读_运行结果.txt',bootstrap['message']+'\n珍珠2固定；人物5保持Reject；未运行Alpha/背景/最终打包。')
        z.writestr('person-session-report.json',redact(json.dumps(bootstrap,ensure_ascii=False,indent=2),os.environ))
    print(bootstrap['message'])
finally:
    if PERSON_ZIP is not None and PERSON_ZIP.exists():
        from IPython.display import display,FileLink
        display(FileLink(str(PERSON_ZIP)))
        try:
            from google.colab import files
            files.download(str(PERSON_ZIP))
        except Exception:print('自动下载被浏览器阻止时，从左侧文件面板下载 FloatDeck-person-coverage-rereview.zip。')
print('本轮只做人物覆盖复审；不运行后续生成。')


## 一键 SAM 分割与预览
**环境已经通过时，直接点击下面单元即可。** 自动复用素材；丢失时才弹出ZIP上传框。结果仅用于审核，不会选择或批准候选。


In [ ]:
#@title ▶ 一键 SAM 分割与预览（环境已通过时，只点这里）
from pathlib import Path
import os,sys,json,subprocess,shutil,re,time,traceback,zipfile
BASE=Path('/content/floatdeck-v4');ROOT=Path('/content/FloatDeck-v4-generator')
ENV_READY=False;ALLOW_ASSET_STAGES=False
bootstrap={'passed':False,'stage':'repository','commands':[],'models_started':False}
def redact(text,env=None):
    text=str(text)
    for k,v in (env or {}).items():
        if re.search(r'token|password|secret|credential|api.?key|authorization',k,re.I) and len(v)>=6:
            text=text.replace(v,'[REDACTED]')
    text=re.sub(r'(https?://)[^\s/@]+:[^\s/@]+@',r'\1[REDACTED]@',text)
    text=re.sub(r'(?i)Bearer\s+[^\s\"\x27]+','Bearer [REDACTED]',text)
    text=re.sub(r'(?i)((?:token|password|secret|api[_-]?key|authorization|signature|credential)[\w-]*[\s\"\x27]*[:=][\s\"\x27]*)([^\s\"\x27,&}]+)',r'\1[REDACTED]',text)
    text=re.sub(r'\b(?:hf_|ghp_|github_pat_)[A-Za-z0-9_]+','[REDACTED]',text)
    return text
def boot(args):
    p=subprocess.run(list(map(str,args)),capture_output=True,text=True,errors='replace')
    bootstrap['commands'].append({'argv':list(map(str,args)),'returncode':p.returncode,'stdout':p.stdout,'stderr':p.stderr})
    if p.returncode:raise subprocess.CalledProcessError(p.returncode,args,p.stdout,p.stderr)
    return p
try:
    BASE.mkdir(parents=True,exist_ok=True)
    print('准备 SAM 分割；不会重新安装依赖或运行抠图/背景补全。')
    if not (ROOT/'.git').exists():
        boot(['git','clone','--branch','spatial-layered-v4-generator','--single-branch','https://github.com/fdsfbgdf-blip/FloatDeck.git',ROOT])
    else:
        if boot(['git','-C',ROOT,'branch','--show-current']).stdout.strip()!='spatial-layered-v4-generator':raise RuntimeError('已有代码位于其他分支，已停止')
        boot(['git','-C',ROOT,'pull','--ff-only','origin','spatial-layered-v4-generator'])
    if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
    import importlib
    import tools.layered_v4.diagnostics as diagnostics
    importlib.reload(diagnostics)
    import tools.layered_v4.sam_export as sam_export
    importlib.reload(sam_export)
    project=BASE/'project-001';python=BASE/'python312/bin/python'
    sam_export.require_environment(project,python)
    ARCHIVE=None
    if not all((project/name).is_file() for name in ['source.png','source_depth.png']):
        print('素材已丢失，请在弹出的上传框选择原始 wallpaper_001.spatial.zip。')
        from google.colab import files
        uploaded=files.upload()
        if len(uploaded)!=1:raise RuntimeError('需要上传一个原始 spatial.zip')
        ARCHIVE=BASE/('sam-input-'+str(time.time_ns())+'.zip');ARCHIVE.write_bytes(next(iter(uploaded.values())))
    result=sam_export.run_export(BASE,ROOT,ARCHIVE,bootstrap['commands'])
    SAM_ZIP=Path(result['zip'])
    from IPython.display import display,Image as DisplayImage
    for path in sorted((Path(result['work'])/'segmentation-previews').glob('*_review_*.jpg')):
        display(DisplayImage(filename=str(path)))
except Exception as exc:
    bootstrap.update(error=str(exc),traceback=traceback.format_exc(),message='分割准备失败，已停止。把下载的 SAM 审核 ZIP 发给 ChatGPT 即可，不需要分析日志。')
    SAM_ZIP=BASE/'FloatDeck-SAM-review.zip'
    with zipfile.ZipFile(SAM_ZIP,'w',zipfile.ZIP_DEFLATED) as z:z.writestr('sam-session-report.json',redact(json.dumps(bootstrap,ensure_ascii=False,indent=2),os.environ))
    print(bootstrap['message'])
finally:
    if 'SAM_ZIP' in globals() and SAM_ZIP.exists():
        from IPython.display import display,FileLink
        display(FileLink(str(SAM_ZIP)))
        try:
            from google.colab import files
            files.download(str(SAM_ZIP))
        except Exception:print('自动下载被浏览器阻止时，从左侧文件面板下载 FloatDeck-SAM-review.zip。')
print('本次停在 SAM 候选审核；无需选择候选编号，不要继续运行后续单元。')


## 环境检查已合并至上方单元
以下为后续素材工作区，本轮不需要操作。模型版本锁定与人工审核仍必需，不会自动放行。


In [ ]:
if not globals().get('ALLOW_ASSET_STAGES',False):raise RuntimeError('本次只检查环境，请停在第一个单元；后续素材阶段尚未开启')
# 后续素材阶段需要模型版本锁；已有锁原样复用，不自动刷新。
if not (WORK/'models.lock.json').exists():run('lock-models')


## 上传实际 ZIP
上传 `wallpaper_001.spatial.zip`。源图/深度保持同尺寸、同方向；遇到不一致或 EXIF 旋转会明确停止，不擅自旋转一张图。项目重跑请更换 WORK，防止覆盖已审核文件。

In [ ]:
if not globals().get('ALLOW_ASSET_STAGES',False):raise RuntimeError('本次只检查环境，请停在第一个单元；后续素材阶段尚未开启')
from google.colab import files
uploaded=files.upload()
if len(uploaded)!=1:raise ValueError('只上传一个输入 ZIP')
INPUT=BASE/'input.spatial.zip';INPUT.write_bytes(next(iter(uploaded.values())))
run('init','--input',INPUT)
from PIL import Image
from IPython.display import display
source=Image.open(WORK/'source.png');print('原始像素尺寸:',source.size)
preview=source.copy();preview.thumbnail((560,1000));display(preview)
print((WORK/'input.json').read_text())


## 分割建议（必须编辑审核）
配置中的框和局部区域基于本轮仙女图。更换图片必须改框。SAM 输出三候选和分数，分数不等于正确；需把飞散发丝、光球旁手指和裙摆纳入人物，珍珠只保留球体，不把岩石或白纱当珍珠。
每次 `run` 是单独进程；SAM 结束后退出，CUDA 内存不与 VitMatte/SDXL 共存。

In [ ]:
if not globals().get('ALLOW_ASSET_STAGES',False):raise RuntimeError('本次只检查环境，请停在第一个单元；后续素材阶段尚未开启')
# 分割修复：旧项目 config 不会自动升级，显式迁移场景提示后再运行。
# 先确认第一单元已拉取最新分支；无需重新上传/初始化已有 source。
import copy
from PIL import Image
from IPython.display import display
from google.colab import files
source=Image.open(WORK/'source.png')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from tools.layered_v4.segmentation import PROMPTS,previews
cfg=json.loads((WORK/'config.json').read_text())
RESET_SCENE_PROMPTS=True # 首次修复旧项目为 True；再次调整后设 False 保留手动提示
if RESET_SCENE_PROMPTS:cfg['sam_prompts']=copy.deepcopy(PROMPTS)
# 可改每个对象的 box / positive / negative，全部是原图归一化 XY。
cfg['boxes']={k:v['box'] for k,v in cfg['sam_prompts'].items()}
(WORK/'config.json').write_text(json.dumps(cfg,ensure_ascii=False,indent=2))
def show_masks(names):
    for name in names:
        path=WORK/name
        if path.exists():
            print('文件 / 图层:',name);im=Image.open(path).copy();im.thumbnail((650,1050));display(im)
previews(WORK,cfg)
show_masks(['source.png','segmentation-previews/person_prompts.png','segmentation-previews/pearl_prompts.png'])
run('segment') # 仅生成候选，不自动选择，不执行任何抠图/补全
for name in ['person','pearl']:
    print('目标:',name, (WORK/f'{name}_proposal.json').read_text())
    for i in range(3):show_masks([f'{name}_candidate_{i}.png',f'segmentation-previews/{name}_candidate_{i}_overlay.png'])


In [ ]:
if not globals().get('ALLOW_ASSET_STAGES',False):raise RuntimeError('本次只检查环境，请停在第一个单元；后续素材阶段尚未开启')
# 逐个查看所有带名称候选；填入索引 0/1/2。无合格候选则调整提示重跑上一单元。
SELECT={'person':None,'pearl':None}
for name,index in SELECT.items():
    if type(index) is not int or index not in range(3):raise RuntimeError('必须人工选择 '+name+' 候选，禁止自动按分数选择')
for name,index in SELECT.items():shutil.copyfile(WORK/f'{name}_candidate_{index}.png',WORK/f'{name}_mask.png')
previews(WORK,cfg)
show_masks(['source.png','person_mask.png','segmentation-previews/person_mask_overlay.png','pearl_mask.png','segmentation-previews/pearl_mask_overlay.png'])
print('尚未人工审核通过；可在下面画笔修正原生尺寸遮罩。')


### 原生坐标编辑器 / 外部软件
白=前景，黑=背景；Trimap 灰=未知，Alpha 灰=半透明。可反复更换 EDIT_FILE 并运行编辑器，缩放后逐笔修正。保存不会自动批准。
也可下载 PNG 在 GIMP/Krita 编辑，然后用下一格上传。必须保持原生宽高，L 灰度 PNG；不要上传缩略图、黑底 RGB 抠图或截图。

In [ ]:
if not globals().get('ALLOW_ASSET_STAGES',False):raise RuntimeError('本次只检查环境，请停在第一个单元；后续素材阶段尚未开启')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from tools.layered_v4.editor import display_editor
EDIT_FILE='person_mask.png'  # pearl_mask.png / 后续的 person_trimap.png、person_alpha.png 等
# 显式使用 Colab Notebook 内核只显示/编辑图片，不在内核加载 GPU 模型。
display_editor(WORK,EDIT_FILE)


In [ ]:
if not globals().get('ALLOW_ASSET_STAGES',False):raise RuntimeError('本次只检查环境，请停在第一个单元；后续素材阶段尚未开启')
# 可选：替换一个或多个经外部编辑的遮罩；不执行时保留编辑器已保存文件。
UPLOAD_EDITED_MASKS=False
if UPLOAD_EDITED_MASKS:
    from tools.layered_v4.editor import ALLOWED
    edited=files.upload()
    for name,data in edited.items():
        if name not in ALLOWED:raise ValueError('文件名不在遮罩白名单: '+name)
        import io
        im=Image.open(io.BytesIO(data))
        if im.size!=source.size or im.mode!='L':raise ValueError('必须保持原生尺寸、8-bit L 通道')
        (WORK/name).write_bytes(data)


## 精细抠图 / 白纱 Alpha
先确认分割，再运行 VitMatte。默认 768 tile / 128 overlap，原生坐标、不缩放原图。配置中的 translucent_regions 是本图白纱不确定区提示，会把支持内该区域设为灰 trimap，绝非透明度真值；更换照片必须修改。三值 trimap 可编辑；脸、眼睛、手指核心及珍珠反光内部应为确定前景。
VitMatte 不是透明白纱的真值。薄纱处需调 `person_alpha.png`，不要把半透明全部涂成白色；保护核心 `person_protect.png` 强制保留原图 RGB。球体内部保持不透明，不把高光当透明。
若修改了已有 trimap，再运行 matte，并同步检查 person_protect.png：旧保护区仍会强制不透明，必要时擦除。若直接修改 alpha，不要再跑 matte 覆盖。

In [ ]:
if not globals().get('ALLOW_ASSET_STAGES',False):raise RuntimeError('本次只检查环境，请停在第一个单元；后续素材阶段尚未开启')
# 本次仅修复 SAM。保持暂停，禁止继续 VitMatte/补全/打包。
CONTINUE_AFTER_SEGMENTATION=False
run('check-segmentation')
SEGMENTATION_REVIEWED=False # 查看命名遮罩、叠加图及所有细节后才可手动确认
if not SEGMENTATION_REVIEWED:raise RuntimeError('请先人工检查分割；本次停在此处')
run('approve-segmentation','--reviewed')
if not CONTINUE_AFTER_SEGMENTATION:raise RuntimeError('本次 SAM 修复暂停后续流程')
run('matte')
run('prepare-masks')
show_masks(['person_alpha.png','pearl_alpha.png','person_protect.png','pearl_protect.png','background_hole.png','person_unobserved.png'])


## 批准当前遮罩，再补全背景
检查所有 Alpha、保护核心、背景 hole 和 `person_unobserved.png`。如果 alpha 修改扩大了支持范围，请同步编辑 hole；或显式删除旧 hole 后重跑 prepare-masks 重新从 alpha 生成。
Hole 内允许生成背景；其外最后逐像素复制原始可见背景，不能被模型改写。不要只为掩盖接缝随意扩大 hole。
人物与珍珠互相遮挡的未知人物区域不会填入珍珠纹理。源图下边缘裁切的珍珠不能凭空补完整球体。
遮罩审核后任何修改都会使哈希失效，必须重新检查批准。

In [ ]:
if not globals().get('ALLOW_ASSET_STAGES',False):raise RuntimeError('本次只检查环境，请停在第一个单元；后续素材阶段尚未开启')
if not globals().get('CONTINUE_AFTER_SEGMENTATION',False):raise RuntimeError('本次 SAM 修复禁止运行后续流程')
MASKS_REVIEWED=False # 编辑并检查后手动设 True
if not MASKS_REVIEWED:raise RuntimeError('遮罩/透明白纱/珍珠轮廓尚未审核')
run('approve-masks','--reviewed')
run('background') # SDXL: global + native tiled refine；一次只驻留这一模型，T4 使用 CPU offload
show_masks(['background.png'])
print((WORK/'inpaint-execution.json').read_text())


## 合成检查：零位移、分别位移、局部放大、黑白底
人物和珍珠从原始 RGB 取色；仅对部分 Alpha 像素进行基于补全背景的 straight-alpha 去污染。该逆问题不唯一，色域裁切/重合误差会记录，不能宣称“数学重合即真实抠图”。
必须检查 background 中没有残留人物、珍珠、裙摆/发丝幽灵；分别位移揭露补全区域。未知人物区域另有棋盘诊断预览，不伪装成已恢复的衣料。

In [ ]:
if not globals().get('ALLOW_ASSET_STAGES',False):raise RuntimeError('本次只检查环境，请停在第一个单元；后续素材阶段尚未开启')
if not globals().get('CONTINUE_AFTER_SEGMENTATION',False):raise RuntimeError('本次 SAM 修复禁止运行后续流程')
run('build')
print((WORK/'quality.json').read_text())
for path in sorted((WORK/'previews').glob('*.png')):
    print(path.name);im=Image.open(path).copy();im.thumbnail((1200,900));display(im)


## 最终人工验收 / 打包
每项都需检查原始尺寸局部：脸/眼睛、手指数量姿态、细发丝、白纱、裙摆、珍珠球形和反光、背景无幽灵、位移露底、黑白底 Alpha。
未通过就编辑遮罩/trimap/Alpha/保护核心，重新批准遮罩、补全和合成；不要勾选虚假的通过。
默认仅导出 UNREVIEWED-DRAFT。已审核实验包也不是 Android ZIP V1，不能直接导入现有 APP。

In [ ]:
if not globals().get('ALLOW_ASSET_STAGES',False):raise RuntimeError('本次只检查环境，请停在第一个单元；后续素材阶段尚未开启')
if not globals().get('CONTINUE_AFTER_SEGMENTATION',False):raise RuntimeError('本次 SAM 修复禁止运行后续流程')
CHECKS={k:False for k in ['segmentation','hair','face_fingers','veil_alpha','skirt','pearl','background_no_ghosts','shift_previews','alpha_on_black_white']}
ACCEPT_UNKNOWN_PERSON_BEHIND_PEARL=False # 仅确实接受已知缺口时，才显式确认；不会生成隐藏人物
FINAL_REVIEWED=all(CHECKS.values())
OUT=BASE/('wallpaper_001.spatial-v2.zip' if FINAL_REVIEWED else 'wallpaper_001.UNREVIEWED.spatial-v2.zip')
if FINAL_REVIEWED:
    flags=['--reviewed']+(['--allow-known-person-occlusion'] if ACCEPT_UNKNOWN_PERSON_BEHIND_PEARL else [])
    run('approve-output',*flags)
run('pack','--output',OUT,*([] if FINAL_REVIEWED else ['--draft']))
subprocess.run([PY,'-m','tools.layered_v4.validate',str(OUT),*([] if FINAL_REVIEWED else ['--allow-draft'])],cwd=ROOT,env=MODEL_ENV,check=True)
files.download(str(OUT))


## 故障与边界
- 找不到 nvidia-smi：以独立解释器 torch.cuda 探测为准，不执行不存在的命令。
- No module named addict：确认 pip 安装与模型命令都用上方 PY，而非 Notebook 内核；重新运行 requirements 安装和 preflight。
- CUDA OOM：停止其他模型/旧进程；保持一次一个 stage；把 matte_tile 降为 512、inpaint_tile 降为 768（overlap 仍小于 tile）、global_long_side 改 896 后重跑；不要降低最终 PNG 尺寸或以模糊/复制替代补全。性能和模型质量仍需实际 T4 执行记录。
- 模型下载失败/访问受限：停止，检查缓存、网络和 Hugging Face 权限；禁止用拷贝原图/传统模糊假装 AI 背景。
- 透明衣料和被珍珠遮住的人物纹理无法仅凭单图唯一求解，完整人物恢复需要额外资料；不能重新生成人物替代。
详细格式与限制见仓库 `SPATIAL_V4_GENERATOR_REPORT.md`、`tools/layered_v4/FORMAT_V2_DRAFT.md`。